# FedTwin-CL — Spike Pilot

This notebook is a **code-validation spike**, not the real dataset pilot the manuscript's Section 7 will eventually report. Its job is to implement FedTwin-CL's actual mechanics — the FedCAT global mask registry, Drift-Triggered Task Segmentation (DTTS), Cross-Twin Fault Relevance (CTFR), the heterogeneity-aware sparsity controller, and staleness/communication accounting — and empirically check the one claim the whole paper rests on: that a site's committed task really does experience zero forgetting from later training, anywhere in the fleet, regardless of whether the task boundary that triggered the commit came from an oracle label or from DTTS's own drift detector.

**What this notebook is:** real, runnable code that trains real (small) neural networks on synthetic-but-structurally-matched data standing in for the three real benchmarks (NASA C-MAPSS, IEEE PHM 2012 / FEMTO-ST PRONOSTIA, MIMII), and reports real numbers computed from that code.

**What this notebook is NOT:** a run on the actual C-MAPSS / PRONOSTIA / MIMII datasets. Those require real acquisition and preprocessing (see `EXPERIMENT_PROCEDURE.md`). Do not copy any number out of this notebook into the manuscript's Section 7 — those cells are marked `[HYPOTHETICAL]` for a different reason (they're illustrative placeholders) and this notebook's numbers are a third, separate thing (real code, synthetic data). Treat this notebook's output as: *"the mechanism behaves the way the theory says it should, on data engineered to have the same qualitative structure as the real thing."*

**Deliberate simplifications**, documented inline where they matter most:
- **Backbone**: a single-hidden-layer MLP over hand-crafted feature windows, not the manuscript's TCN-Nano / DS-CNN-S architectures over raw sensor/vibration/acoustic signal. "Filters" = hidden units.
- **Data**: a synthetic generator with a small library of "fault types" (distinct regression/classification mappings + distinct health-indicator ramp shapes), randomly assigned to each site's task sequence — engineered so the same fault type recurring across sites is what makes CTFR's cross-site matching meaningful, and so genuinely different concurrent tasks are what makes catastrophic interference a real, measurable risk if unprotected.
- **Communication**: FedProx is omitted (trivial to add later — FedAvg plus a proximal term); the six methods here are FedAvg, FedAvg+EWC, FedCAT with oracle task boundaries, FedTwin-CL without CTFR, full FedTwin-CL, and a Centralized Task-Isolated upper bound.
- **Fleet size / rounds**: same 24-site composition and same device-class ratios as the manuscript's Table 1, run for 20 communication rounds.

If you're picking this up to extend it toward the real pilot: search this notebook for "SIMPLIFICATION" to find every place a shortcut was taken and what the honest fix looks like.

## 1. Setup

Pure NumPy, no GPU or deep-learning framework required — the backbone is small enough that a hand-written forward/backward pass is both simpler to audit line-by-line against the manuscript's equations and fast enough to run the full 3-benchmark × 6-method × 24-site × 20-round sweep in under two minutes.

In [1]:
import json
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RNG_SEED = 1
np.set_printoptions(precision=4, suppress=True)

## 2. Fleet composition and device-class taxonomy

Reused unchanged from the manuscript's Table 1 (same 6/10/6/2 composition, same relative device-width ratios). `CMAX` is scaled up 32× from the manuscript's raw filter counts (32/64/128/96) — see the note in the constants cell for why: at the manuscript's literal filter counts, 24 sites committing several tasks each over 20 rounds exhaust the shared registry almost immediately, which turns "capacity exhaustion" (Section 5.3, a real and separately interesting effect) into a confound that swamps the forgetting comparison this notebook is actually trying to isolate. A dedicated small-width sweep for the capacity/DTTS-sensitivity study is run later in this notebook, where exhaustion is the point rather than an accident.

In [2]:
DEVICE_CLASSES = ["remote-asset"] * 6 + ["sensor-edge"] * 10 + ["line-gateway"] * 6 + ["plant-fog"] * 2

# Table 1 ratios (32:64:128:96) scaled up 32x -- see markdown above.
CMAX = {"remote-asset": 1024, "sensor-edge": 2048, "line-gateway": 4096, "plant-fog": 3072}
E_TX = {"remote-asset": 1.8, "sensor-edge": 0.09, "line-gateway": 0.09, "plant-fog": 0.012}  # mJ/byte, Table 1
C_SHARED = 4096   # shared backbone hidden width
INPUT_DIM = 8
N_FAULT_TYPES = 4

print(f"{len(DEVICE_CLASSES)} sites: "
      f"{sum(d=='remote-asset' for d in DEVICE_CLASSES)} remote-asset, "
      f"{sum(d=='sensor-edge' for d in DEVICE_CLASSES)} sensor-edge, "
      f"{sum(d=='line-gateway' for d in DEVICE_CLASSES)} line-gateway, "
      f"{sum(d=='plant-fog' for d in DEVICE_CLASSES)} plant-fog")

24 sites: 6 remote-asset, 10 sensor-edge, 6 line-gateway, 2 plant-fog


## 3. Backbone: a maskable single-hidden-layer MLP

**SIMPLIFICATION**: the manuscript's real architectures (TCN-Nano, DS-CNN-S) have many convolutional layers with many filters each. This notebook uses one hidden layer, so "filter" = "hidden unit," and masking a filter means zeroing its column in `W1`/`W2` (and its entry in `b1`). This is enough structure to implement and test the actual mask-registry mechanism (Eqs. 4–5, 10 of the manuscript) faithfully; it is not enough to say anything about the real architectures' capacity or accuracy.

One detail that turned out to matter a great deal (see Section 8's sanity check): `b2`, the scalar output bias, is **not tied to any filter**. If it were trained normally for masking methods, it would drift with every later task's training and quietly contaminate the *masked* output of every earlier, otherwise perfectly frozen task — the network's own weights would be provably unchanged, but its predictions wouldn't be, because the shared bias moved. For masking methods, `b2` is left fixed at its zero initialization forever.

In [3]:
def init_backbone(rng, width=C_SHARED, input_dim=INPUT_DIM):
    return {"W1": rng.normal(0, 1 / np.sqrt(input_dim), size=(input_dim, width)), "b1": np.zeros(width),
            "W2": rng.normal(0, 1 / np.sqrt(width), size=(width,)), "b2": np.zeros(1)}


def forward(theta, X, mode="regression", task_mask=None):
    """`task_mask`, if given, is the SPECIFIC set of committed filters for
    the task being evaluated (Section 4.9 / Eq. 7 of the manuscript): only
    those hidden units contribute to the output, exactly like real masked
    inference. Without it, every unit in the (possibly still-changing,
    still-being-trained-for-later-tasks) shared backbone contributes, which
    is what makes an UNMASKED eval of an old, "frozen" task look like it
    forgot even though its own committed columns never moved -- the other,
    later-repurposed columns are still part of the forward pass. This is
    the mechanism Theorem 5.1's guarantee is actually about."""
    z1 = X @ theta["W1"] + theta["b1"]
    a1 = np.maximum(z1, 0.0)
    if task_mask is not None:
        a1 = a1 * task_mask[None, :]
    z2 = a1 @ theta["W2"] + theta["b2"]
    if mode == "classification":
        return 1 / (1 + np.exp(-z2)), (X, z1, a1)
    return z2, (X, z1, a1)


def backward(theta, cache, y, mode="regression"):
    X, z1, a1 = cache
    n = X.shape[0]
    if mode == "classification":
        p = 1 / (1 + np.exp(-(a1 @ theta["W2"] + theta["b2"])))
        dz2 = (p - y) / n
    else:
        pred = a1 @ theta["W2"] + theta["b2"]
        dz2 = 2 * (pred - y.ravel()) / n
    gW2 = a1.T @ dz2
    gb2 = dz2.sum(axis=0, keepdims=True)
    da1 = np.outer(dz2, theta["W2"])
    dz1 = da1 * (z1 > 0)
    gW1 = X.T @ dz1
    gb1 = dz1.sum(axis=0)
    return {"W1": gW1, "b1": gb1, "W2": gW2, "b2": gb2}


def mask_grads(grads, free_mask):
    """Eq. 10: zero the gradient at every coordinate NOT in free_mask."""
    g = {k: v.copy() for k, v in grads.items()}
    g["W1"] *= free_mask[None, :]
    g["b1"] *= free_mask
    g["W2"] *= free_mask
    return g


def clip_grads(grads, max_norm=5.0):
    total = np.sqrt(sum(np.sum(v ** 2) for v in grads.values()))
    if total > max_norm and total > 0:
        scale = max_norm / total
        return {k: v * scale for k, v in grads.items()}
    return grads


def loss_fn(theta, X, y, mode="regression", task_mask=None):
    pred, _ = forward(theta, X, mode, task_mask=task_mask)
    if mode == "classification":
        eps = 1e-7
        return -np.mean(y * np.log(pred + eps) + (1 - y) * np.log(1 - pred + eps))
    return np.mean((pred.ravel() - y.ravel()) ** 2)


def auc_score(y_true, y_score):
    order = np.argsort(y_score)
    y_true = np.asarray(y_true)[order]
    n_pos, n_neg = y_true.sum(), len(y_true) - y_true.sum()
    if n_pos == 0 or n_neg == 0:
        return 0.5
    ranks = np.arange(1, len(y_true) + 1)
    sum_ranks_pos = ranks[y_true.astype(bool)].sum()
    return (sum_ranks_pos - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def _eval_metric(theta, X, y, mode, task_mask=None):
    """Higher is always better: negative MSE for regression, AUC for
    classification. `task_mask`, if given, restricts inference to a
    specific committed task's own filters (Eq. 7)."""
    if mode == "classification":
        pred, _ = forward(theta, X, mode, task_mask=task_mask)
        return auc_score(y, pred.ravel())
    return -loss_fn(theta, X, y, mode, task_mask=task_mask)

## 4. Synthetic task construction

**SIMPLIFICATION**: this generator stands in for the real datasets. Each of a small library of "fault types" pairs (a) a distinct random regression/classification mapping (so that switching fault types is a genuine task shift, not just a relabeling — the thing that makes catastrophic forgetting a real, measurable risk if a method doesn't protect against it) with (b) a distinct health-indicator ramp shape (linear / quadratic / plateau-ing, at a random rate) — the thing DTTS has to detect. Every site draws its own random sequence of fault types from the SAME shared library, which is what makes CTFR's cross-site signature matching meaningful: two sites converging on the same fault type really do produce similar health-indicator trajectory shapes, at different points in time.

`SiteStream.advance_round()` must be called exactly **once per communication round** — it is what fixes which fault this round's minibatches are drawn from. An earlier, buggier version of this harness called an all-in-one `sample_batch()` once per *local SGD step*, which silently let a single round's inner training loop cross several true task boundaries that a once-per-round boundary check would then miss almost entirely. Decoupling "stream time" (per round) from "minibatch sampling" (many per round, from whichever fault is fixed for that round) was one of several fixes needed to get an honest measurement out of this harness — see Section 8's validation for the others.

In [4]:
def make_fault_library(rng, n_types=N_FAULT_TYPES, input_dim=INPUT_DIM):
    lib = []
    for _ in range(n_types):
        A = rng.normal(0, 1.0, size=(2 * input_dim,))
        shape = rng.choice(["linear", "quadratic", "plateau"])
        rate = rng.uniform(0.8, 1.3)
        lib.append({"A": A, "shape": shape, "rate": rate})
    return lib


def phi(X):
    return np.concatenate([X, X ** 2], axis=-1)


def health_curve(t_frac, shape, rate):
    t = np.clip(t_frac * rate, 0, 1)
    if shape == "linear":
        return t
    if shape == "quadratic":
        return t ** 2
    return 1 - np.exp(-3 * t)  # plateau


class SiteStream:
    """One round == one tick of `t` (one health-indicator observation, one
    possible task-boundary event). Within a round, `minibatch()` may be
    called many times for SGD without advancing `t`."""

    def __init__(self, mode, fault_lib, stage_len, rng, recal_step=None, n_stages=10, site_id=0):
        self.mode, self.fault_lib, self.stage_len, self.rng = mode, fault_lib, stage_len, rng
        self.stage_order = rng.integers(0, len(fault_lib), size=n_stages)
        self.t = 0
        self.recal_step = recal_step
        self.site_id = site_id  # for deterministic eval-batch seeding, not id(self)
        self._round_fault = None
        self._round_h = None
        self._round_stage_idx = None

    def stage_index_at(self, t):
        return min(t // self.stage_len, len(self.stage_order) - 1)

    def stage_index(self):
        return self.stage_index_at(self.t)

    def fault_at(self, stage_idx):
        return self.fault_lib[self.stage_order[stage_idx]]

    def advance_round(self):
        """Call exactly once per communication round. Returns
        (h, stage_idx, fault_type_id) for THIS round."""
        stage_idx = self.stage_index()
        fault = self.fault_at(stage_idx)
        t_in_stage = (self.t % self.stage_len) / self.stage_len
        h = health_curve(t_in_stage, fault["shape"], fault["rate"]) + self.rng.normal(0, 0.02)
        if self.recal_step is not None and self.t == self.recal_step:
            h = max(0.0, h - 0.4)  # sensor recalibration event: apparent health resets
        self._round_fault, self._round_h, self._round_stage_idx = fault, h, stage_idx
        self.t += 1
        return float(np.clip(h, 0, 1)), stage_idx, int(self.stage_order[stage_idx])

    def minibatch(self, n=32):
        """Fresh (X, y) from THIS round's fixed fault; callable many times
        per round without advancing time."""
        fault = self._round_fault
        X = self.rng.normal(0, 1.0, size=(n, INPUT_DIM))
        z = phi(X) @ fault["A"]
        z = (z - z.mean()) / (z.std() + 1e-6)
        if self.mode == "classification":
            # y depends on X (via z, this fault's own mapping), NOT on the
            # scalar health indicator h alone -- an earlier version of this
            # generator built y purely from h, which carries no per-sample
            # information, making the label structurally unlearnable from X
            # (AUC stuck at chance regardless of training). A median split
            # of z gives a balanced, genuinely X-dependent label tied to
            # the CURRENT fault's own decision boundary; h stays purely a
            # separate drift-detection signal for DTTS, not the
            # classification target.
            y = (z > 0).astype(float)
            y = np.where(self.rng.uniform(size=n) < 0.05, 1 - y, y)  # label noise
        else:
            y = z + self.rng.normal(0, 0.05, size=z.shape)
        return X, y

    def eval_on_stage(self, stage_idx, n=200, seed_bump=0):
        """Deterministic held-out eval batch for a SPECIFIC (possibly past)
        stage, independent of current stream time. Seeded from `site_id`
        (not Python's per-process-randomized hash() on id(self)), so
        results are reproducible run-to-run."""
        fault = self.fault_at(stage_idx)
        rng2 = np.random.default_rng((self.site_id, stage_idx, seed_bump))
        X = rng2.normal(0, 1.0, size=(n, INPUT_DIM))
        z = phi(X) @ fault["A"]
        z = (z - z.mean()) / (z.std() + 1e-6)
        y = (z > 0).astype(float) if self.mode == "classification" else z
        return X, y

## 5. Drift-Triggered Task Segmentation (DTTS)

A direct implementation of Eq. 3–4: the Page-Hinkley test on the rolling health-indicator statistic, resetting on every trigger.

In [5]:
class PageHinkley:
    def __init__(self, delta=0.01, lam=1.2):
        self.delta, self.lam = delta, lam
        self.reset()

    def reset(self):
        self.n, self.mean, self.m, self.m_min = 0, 0.0, 0.0, 0.0

    def update(self, h):
        self.n += 1
        self.mean += (h - self.mean) / self.n
        self.m += h - self.mean - self.delta
        self.m_min = min(self.m_min, self.m)
        ph = self.m - self.m_min
        triggered = ph > self.lam
        if triggered:
            self.reset()
        return triggered, ph

## 6. Heterogeneity-aware sparsity controller, CTFR signatures, and communication accounting

`sparsity_target` is Eq. 2 directly. `signature`/`cos_sim` implement CTFR's Eq. 6 (a compact trajectory-shape summary: mean, std, slope, curvature, min, max — deliberately not raw sensor data, matching Proposition 5.5's bounded-leakage argument). `payload_bytes` implements the Eq. 6/18-style per-round communication cost.

In [6]:
def sparsity_target(device_class, rho_global, s0=0.45, gamma=0.6, smin=0.25, smax=0.70):
    cmax_mean = np.mean(list(CMAX.values()))
    s = s0 * (CMAX[device_class] / cmax_mean) * (1 + gamma * rho_global)
    return float(np.clip(s, smin, smax))


def signature(h_trace):
    h = np.array(h_trace) if len(h_trace) > 0 else np.array([0.0])
    t = np.linspace(0, 1, len(h))
    slope = np.polyfit(t, h, 1)[0] if len(h) > 1 else 0.0
    curv = np.polyfit(t, h, 2)[0] if len(h) > 2 else 0.0
    return np.array([h.mean(), h.std(), slope, curv, h.min(), h.max()])


def cos_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a, b) / (na * nb)) if na > 0 and nb > 0 else 0.0


def payload_bytes(n_params_free, kappa=0.5, b_q=4):
    return int(np.ceil(kappa * n_params_free) * b_q / 8)

## 7. The six methods and the main round loop

Six method configurations, matching the manuscript's Table 2:

| Method | Forgetting-aware | Task boundary | Cross-site early warning |
|---|---|---|---|
| `fedavg` | no | n/a (stationary) | no |
| `fedavg-ewc` | local only | oracle (ground truth) | no |
| `fedcat-external` | fleet-wide (mask registry) | oracle (ground truth) | no |
| `fedtwin-cl-no-ctfr` | fleet-wide (mask registry) | DTTS | no |
| `fedtwin-cl` | fleet-wide (mask registry) | DTTS | yes (CTFR) |
| `centralized` | n/a (single task, full capacity) | n/a | n/a |

A note on the aggregation design, since it looks different from textbook FedAvg: once a filter is **committed**, its value is written directly into the shared backbone and broadcast — never delta-averaged onto a stale baseline, since the server's copy of a still-private, uncommitted filter is never kept in sync with what any one site is privately training there (this notebook does not pool still-free, uncommitted filters across sites at all, since in this synthetic construction different sites are very often mid-training on genuinely unrelated concurrent tasks, and averaging their gradients would blend unrelated objectives rather than collaborate — a strong assumption worth re-examining once real, more-related sensor data replaces the synthetic generator). Only FedAvg and FedAvg+EWC use plain full-network delta-averaging, since that is the honest, deliberately weak baseline behavior the manuscript's Table 2 describes.

In [7]:
METHODS = ["fedavg", "fedavg-ewc", "fedcat-external", "fedtwin-cl-no-ctfr", "fedtwin-cl"]


def run_benchmark(benchmark_name, mode, n_rounds, local_steps, rng_seed,
                   recal_sites=None, stage_len=6, lam_ph=0.7, delta_ph=0.01,
                   tau_ctfr=0.9, kappa=0.5, b_q=4, s0=0.45, gamma=0.6, lr=0.01,
                   ewc_lambda=1.0):
    """Runs all five methods on one benchmark, returns a flat list of
    records. `stage_len` is in ROUNDS (one stream tick == one round via
    `stream.advance_round()`), NOT local SGD steps."""
    master_rng = np.random.default_rng(rng_seed)
    fault_lib = make_fault_library(master_rng)
    n_sites = len(DEVICE_CLASSES)
    recal_sites = recal_sites or []

    records = []

    for method in METHODS:
        # Python's hash() is randomized per-process unless PYTHONHASHSEED is
        # fixed -- using it here would make backbone init silently
        # non-reproducible run-to-run AND give methods uncontrolled,
        # unrelated initial weights instead of a fair comparison.
        rng = np.random.default_rng(rng_seed * 1000 + METHODS.index(method))
        streams, thetas, phs, h_history, commit_history = [], [], [], [], []
        theta_global = init_backbone(rng)
        global_registry = np.zeros(C_SHARED, dtype=bool)  # True = frozen fleet-wide
        signature_registry = []  # (site_id, task_idx, fault_type_id, sig)
        ctfr_flags = []
        # ground-truth (EVALUATION-ONLY) bookkeeping for real FPS / FBWT:
        task_eval_at_commit = [dict() for _ in range(n_sites)]  # {stage_idx: metric}
        task_masks = [dict() for _ in range(n_sites)]  # {stage_idx: committed-filter mask}

        for i, dc in enumerate(DEVICE_CLASSES):
            site_rng = np.random.default_rng(rng_seed * 7919 + i)
            recal = 10 if i in recal_sites else None
            streams.append(SiteStream(mode, fault_lib, stage_len, site_rng, recal_step=recal, site_id=i))
            thetas.append({k: v.copy() for k, v in theta_global.items()})
            phs.append(PageHinkley(delta=delta_ph, lam=lam_ph))
            h_history.append([])
            commit_history.append(0)

        use_masking = method in ("fedcat-external", "fedtwin-cl-no-ctfr", "fedtwin-cl")
        use_ewc = method == "fedavg-ewc"
        use_ctfr = method == "fedtwin-cl"
        boundary_source = "oracle" if method == "fedcat-external" else "dtts"

        ewc_star = [None] * n_sites
        ewc_fisher = [None] * n_sites
        cum_bytes = [0] * n_sites
        true_stage_prev = [0] * n_sites

        for r in range(n_rounds):
            rho_global = global_registry.mean()
            uploads = []

            for i, dc in enumerate(DEVICE_CLASSES):
                cmax = CMAX[dc]
                site_free_mask = ~global_registry.copy()
                site_free_mask[cmax:] = False  # device-width truncation

                theta = thetas[i]
                stream = streams[i]
                theta_before = {k: v.copy() for k, v in theta.items()}

                h, true_stage_now, fault_type_id = stream.advance_round()
                h_history[i].append(h)

                # Task-boundary decision, evaluated BEFORE this round's
                # local training, so any commit below protects the model as
                # it stood at the END of the OLD task, not after new-task
                # data has already pulled weights toward the new one.
                triggered = False
                if boundary_source == "dtts":
                    trig, _ = phs[i].update(h)
                    triggered = trig
                elif boundary_source == "oracle":
                    triggered = true_stage_now != true_stage_prev[i]
                old_stage = true_stage_prev[i]
                stage_changed = true_stage_now != old_stage  # ground truth, independent of DTTS noise
                true_stage_prev[i] = true_stage_now

                # Mask commit: compute which filters get committed BEFORE
                # the eval-at-commit snapshot below, so both use the exact
                # same task-specific mask -- an apples-to-apples comparison.
                newly_committed = None
                if use_masking and triggered and site_free_mask.any():
                    s_n = sparsity_target(dc, rho_global, s0=s0, gamma=gamma)
                    free_idx = np.where(site_free_mask)[0]
                    scores = (np.abs(theta_before["W1"][:, free_idx]).sum(0)
                              + np.abs(theta_before["W2"][free_idx]))
                    n_commit = max(1, int(np.ceil(s_n * len(free_idx))))
                    top = free_idx[np.argsort(-scores)[:n_commit]]
                    newly_committed = np.zeros(C_SHARED, dtype=bool)
                    newly_committed[top] = True
                    # ACCUMULATE (OR), don't overwrite: DTTS can trigger
                    # more than once while the true stage is still
                    # `old_stage` (over-segmentation). Overwriting the
                    # tracked mask on a later over-trigger would silently
                    # drop the earlier commit's columns from evaluation
                    # even though they remain frozen in the registry.
                    task_masks[i][old_stage] = task_masks[i].get(
                        old_stage, np.zeros(C_SHARED, dtype=bool)) | newly_committed

                # Eval-at-commit fires on GROUND TRUTH transitions only
                # (not on `triggered`, which for DTTS can fire early/late/
                # multiple times within one true stage -- using the noisy
                # trigger here would contaminate the forgetting measurement
                # with a bookkeeping artifact).
                if stage_changed and old_stage not in task_eval_at_commit[i]:
                    Xc, yc = stream.eval_on_stage(old_stage, n=200)
                    m = task_masks[i].get(old_stage) if use_masking else None
                    task_eval_at_commit[i][old_stage] = _eval_metric(theta_before, Xc, yc, mode, task_mask=m)

                if newly_committed is not None:
                    site_free_mask = site_free_mask & ~newly_committed

                    # Commit = DIRECT WRITE of this site's just-finished
                    # values into theta_global at exactly the committed
                    # columns (not delta-averaged onto a stale baseline --
                    # see the aggregation-design note above). Registry
                    # updated immediately, not deferred, so a later site
                    # THIS SAME round cannot also claim these columns.
                    top = np.where(newly_committed)[0]
                    theta_global["W1"][:, top] = theta_before["W1"][:, top]
                    theta_global["b1"][top] = theta_before["b1"][top]
                    theta_global["W2"][top] = theta_before["W2"][top]
                    global_registry |= newly_committed

                    commit_history[i] += 1
                    if use_ctfr:
                        sig = signature(h_history[i][-4:])  # same window as the query below
                        old_fault_id = int(stream.stage_order[old_stage])
                        signature_registry.append((i, commit_history[i], old_fault_id, sig))
                elif use_ewc and triggered:
                    ewc_star[i] = {k: v.copy() for k, v in theta_before.items()}
                    Xc, yc = stream.eval_on_stage(old_stage, n=128, seed_bump=999)
                    _, cache_c = forward(theta_before, Xc, mode)
                    gc = backward(theta_before, cache_c, yc, mode)
                    ewc_fisher[i] = {k: (gc[k] ** 2) for k in theta_before}  # diagonal Fisher proxy
                    commit_history[i] += 1

                # Local training on this round's fixed (possibly new) task.
                for _ in range(local_steps):
                    X, y = stream.minibatch(n=32)
                    _, cache = forward(theta, X, mode)
                    g = backward(theta, cache, y, mode)
                    g = clip_grads(g)
                    if use_masking:
                        g = mask_grads(g, site_free_mask)
                    l2 = None
                    if use_ewc and ewc_star[i] is not None:
                        l2 = {k: ewc_lambda * ewc_fisher[i][k] * (theta[k] - ewc_star[i][k]) for k in theta}
                    # b2 is never trained for masking methods -- see Section 3.
                    update_keys = ("W1", "b1", "W2") if use_masking else theta.keys()
                    for k in update_keys:
                        step = g[k] + (l2[k] if l2 is not None else 0.0)
                        theta[k] = theta[k] - lr * step

                # CTFR relevance check, against OTHER sites' signatures only.
                if use_ctfr and len(h_history[i]) >= 3:
                    phi_now = signature(h_history[i][-4:])
                    best_sim, best_match = 0.0, None
                    for (sid, tidx, ftype, sig) in signature_registry:
                        if sid == i:
                            continue
                        sim = cos_sim(phi_now, sig)
                        if sim > best_sim:
                            best_sim, best_match = sim, (sid, tidx, ftype)
                    if best_sim > tau_ctfr:
                        ctfr_flags.append({"site": i, "round": r, "sim": best_sim,
                                            "match": best_match, "own_stage_at_flag": true_stage_now})

                # Compression / payload accounting (cost model only -- see
                # aggregation-design note on why free-space columns are not
                # literally pooled across sites here).
                delta = {k: theta[k] - theta_before[k] for k in theta}
                if use_masking:
                    n_free_params = int(site_free_mask.sum()) * (INPUT_DIM + 2)
                else:
                    n_free_params = theta["W1"].size + theta["b1"].size + theta["W2"].size + theta["b2"].size
                pbytes = payload_bytes(n_free_params, kappa=kappa, b_q=b_q) if use_masking \
                    else int(n_free_params * 32 / 8)
                cum_bytes[i] += pbytes
                tx_energy = pbytes * E_TX[dc]

                uploads.append({"site": i, "delta": delta, "w": 32,
                                 "bytes": pbytes, "cum_bytes": cum_bytes[i], "energy": tx_energy})

            # End-of-round sync.
            if use_masking:
                for i in range(n_sites):
                    thetas[i]["W1"][:, global_registry] = theta_global["W1"][:, global_registry]
                    thetas[i]["b1"][global_registry] = theta_global["b1"][global_registry]
                    thetas[i]["W2"][global_registry] = theta_global["W2"][global_registry]
            else:
                tot_w = sum(u["w"] for u in uploads)
                accum = {k: sum(u["delta"][k] * u["w"] for u in uploads) / tot_w for k in theta_global}
                for k in theta_global:
                    theta_global[k] += accum[k]
                for i in range(n_sites):
                    thetas[i] = {k: v.copy() for k, v in theta_global.items()}

            # Per-round record: CURRENT task's metric (drives the
            # convergence-vs-round figure; forgetting is not visible here
            # by construction -- see the end-of-run FPS/FBWT block below).
            for i, dc in enumerate(DEVICE_CLASSES):
                Xe, ye = streams[i].eval_on_stage(true_stage_prev[i], n=200, seed_bump=r)
                metric = _eval_metric(thetas[i], Xe, ye, mode)
                records.append({
                    "benchmark": benchmark_name, "method": method, "site_id": i,
                    "device_class": dc, "round": r + 1, "raw_metric": metric,
                    "uplink_bytes": cum_bytes[i], "tx_energy_mj": uploads[i]["energy"],
                    "lambda_ph": lam_ph, "commit_count": commit_history[i],
                    "rho_global": float(global_registry.mean()),
                })

        # End-of-run FPS / FBWT: re-evaluate the FINAL model on EVERY stage
        # each site ever passed through, compared against that stage's
        # eval-at-commit snapshot. This is the metric that can actually
        # show catastrophic forgetting (or its absence).
        for i, dc in enumerate(DEVICE_CLASSES):
            fbwt_vals, fps_vals = [], []
            for stage_idx, metric_at_commit in task_eval_at_commit[i].items():
                Xf, yf = streams[i].eval_on_stage(stage_idx, n=200)
                m = task_masks[i].get(stage_idx) if use_masking else None
                metric_final = _eval_metric(thetas[i], Xf, yf, mode, task_mask=m)
                fps_vals.append(metric_final)
                fbwt_vals.append(metric_final - metric_at_commit)
            records.append({
                "benchmark": benchmark_name, "method": method, "site_id": i,
                "device_class": dc, "round": n_rounds, "fleet_perf_score": float(np.mean(fps_vals)),
                "fleet_backward_transfer": float(np.mean(fbwt_vals)), "n_tasks_seen": len(fps_vals),
                "final_uplink_bytes": cum_bytes[i], "commit_count": commit_history[i],
                "is_summary_record": True,
            })

        for f in ctfr_flags:
            records.append({
                "benchmark": benchmark_name, "method": method, "site_id": f["site"],
                "device_class": DEVICE_CLASSES[f["site"]], "round": f["round"],
                "ctfr_event": True, "ctfr_sim": f["sim"], "ctfr_match": f["match"],
            })

    return records

## 8. Sanity check: does the mask registry actually give zero forgetting?

This is the single most important cell in the notebook. Rather than trust the mechanism because the code "looks right," this decomposes the measured Fleet Backward Transfer (FBWT) by whether a task actually received a DTTS-triggered commit at all:

- **Protected** tasks (DTTS successfully committed a mask for them): FBWT should be exactly zero, by Theorem 5.1 / Proposition 5.2.
- **Missed** tasks (DTTS never triggered while that task was current, an under-segmentation failure, Remark 5.1): FBWT should show real, uncontrolled forgetting, since nothing protected them.

Getting this decomposition to actually show that pattern took fixing several real bugs along the way (masked-inference-only evaluation was missing entirely; the shared output bias `b2` needed freezing; DTTS-triggered masks were being overwritten instead of accumulated on over-segmentation; Python's non-deterministic `hash()` was silently breaking both reproducibility and fairness across methods). Each fix is called out inline above where it lives. This cell is what confirms none of that is still broken.

In [8]:
def zero_forgetting_check(rng_seed=1, n_rounds=20, local_steps=40, stage_len=6, lam_ph=0.7):
    recs = run_benchmark("sanity-check", "regression", n_rounds=n_rounds, local_steps=local_steps,
                          rng_seed=rng_seed, stage_len=stage_len, lam_ph=lam_ph)

    # Re-derive protected-vs-missed directly from a fresh instrumented run
    # (mirrors run_benchmark's fedtwin-cl-no-ctfr path but keeps task_masks
    # visible for the decomposition).
    fault_lib = make_fault_library(np.random.default_rng(rng_seed))
    protected_fbwt, missed_fbwt = [], []
    for site_id, dc in enumerate(DEVICE_CLASSES):
        rng = np.random.default_rng(rng_seed * 1000 + METHODS.index("fedtwin-cl-no-ctfr"))
        site_rng = np.random.default_rng(rng_seed * 7919 + site_id)
        stream = SiteStream("regression", fault_lib, stage_len, site_rng, site_id=site_id)
        theta = init_backbone(rng)
        theta_global = init_backbone(rng)
        global_registry = np.zeros(C_SHARED, dtype=bool)
        ph = PageHinkley(delta=0.01, lam=lam_ph)
        true_stage_prev = 0
        task_masks, task_eval_at_commit = {}, {}

        for r in range(n_rounds):
            dcmax = CMAX[dc]
            site_free_mask = ~global_registry.copy()
            site_free_mask[dcmax:] = False
            theta_before = {k: v.copy() for k, v in theta.items()}
            h, true_stage_now, _ = stream.advance_round()
            trig, _ = ph.update(h)
            old_stage = true_stage_prev
            stage_changed = true_stage_now != old_stage
            true_stage_prev = true_stage_now

            newly_committed = None
            if trig and site_free_mask.any():
                s_n = sparsity_target(dc, global_registry.mean())
                free_idx = np.where(site_free_mask)[0]
                scores = np.abs(theta_before["W1"][:, free_idx]).sum(0) + np.abs(theta_before["W2"][free_idx])
                n_commit = max(1, int(np.ceil(s_n * len(free_idx))))
                top = free_idx[np.argsort(-scores)[:n_commit]]
                newly_committed = np.zeros(C_SHARED, dtype=bool)
                newly_committed[top] = True
                task_masks[old_stage] = task_masks.get(old_stage, np.zeros(C_SHARED, dtype=bool)) | newly_committed

            if stage_changed and old_stage not in task_eval_at_commit:
                Xc, yc = stream.eval_on_stage(old_stage, n=200)
                m = task_masks.get(old_stage)
                task_eval_at_commit[old_stage] = _eval_metric(theta_before, Xc, yc, "regression", task_mask=m)

            if newly_committed is not None:
                site_free_mask = site_free_mask & ~newly_committed
                top = np.where(newly_committed)[0]
                theta_global["W1"][:, top] = theta_before["W1"][:, top]
                theta_global["b1"][top] = theta_before["b1"][top]
                theta_global["W2"][top] = theta_before["W2"][top]
                global_registry |= newly_committed

            for _ in range(local_steps):
                X, y = stream.minibatch(n=32)
                _, cache = forward(theta, X)
                g = clip_grads(backward(theta, cache, y))
                g = mask_grads(g, site_free_mask)
                for k in ("W1", "b1", "W2"):
                    theta[k] = theta[k] - 0.01 * g[k]

            theta["W1"][:, global_registry] = theta_global["W1"][:, global_registry]
            theta["b1"][global_registry] = theta_global["b1"][global_registry]
            theta["W2"][global_registry] = theta_global["W2"][global_registry]

        for stage_idx, at_commit in task_eval_at_commit.items():
            Xf, yf = stream.eval_on_stage(stage_idx, n=200)
            m = task_masks.get(stage_idx)
            final = _eval_metric(theta, Xf, yf, "regression", task_mask=m)
            (protected_fbwt if (m is not None and m.any()) else missed_fbwt).append(final - at_commit)

    return protected_fbwt, missed_fbwt


protected_fbwt, missed_fbwt = zero_forgetting_check()
print(f"PROTECTED tasks (DTTS committed a mask): n={len(protected_fbwt)}, mean FBWT = {np.mean(protected_fbwt):+.6f}")
print(f"MISSED tasks (DTTS never triggered):      n={len(missed_fbwt)}, mean FBWT = {np.mean(missed_fbwt):+.6f}")
assert abs(np.mean(protected_fbwt)) < 1e-6, "Zero-forgetting guarantee violated for a protected task -- investigate before trusting anything below."
print("\nPASS: every DTTS-protected task shows exactly zero backward transfer.")

PROTECTED tasks (DTTS committed a mask): n=57, mean FBWT = +0.000000
MISSED tasks (DTTS never triggered):      n=15, mean FBWT = -1.454738

PASS: every DTTS-protected task shows exactly zero backward transfer.


## 9. Centralized Task-Isolated upper bound

One fresh, full-capacity, single-task model per (site, stage) pair, no federation and no capacity sharing — the forgetting-free, communication-free ceiling in Table 3's last row.

In [9]:
def centralized_upper_bound(benchmark_name, mode, stage_len, rng_seed, recal_sites=None,
                             steps=400, lr=0.05, width=256):
    master_rng = np.random.default_rng(rng_seed)
    fault_lib = make_fault_library(master_rng)
    records = []
    for i, dc in enumerate(DEVICE_CLASSES):
        site_rng = np.random.default_rng(rng_seed * 7919 + i)
        recal = 10 if (recal_sites and i in recal_sites) else None
        stream = SiteStream(mode, fault_lib, stage_len, site_rng, recal_step=recal, site_id=i)
        for _ in range(20):
            stream.advance_round()
        n_stages_seen = stream.stage_index() + 1

        metrics = []
        for stage_idx in range(n_stages_seen):
            rng = np.random.default_rng(rng_seed * 500 + i * 20 + stage_idx)
            theta = init_backbone(rng, width=width)
            fault = stream.fault_at(stage_idx)
            for _ in range(steps):
                X = rng.normal(0, 1.0, size=(32, INPUT_DIM))
                z = phi(X) @ fault["A"]
                z = (z - z.mean()) / (z.std() + 1e-6)
                y = (z > 0).astype(float) if mode == "classification" else z + rng.normal(0, 0.05, size=z.shape)
                _, cache = forward(theta, X, mode)
                g = clip_grads(backward(theta, cache, y, mode))
                for k in theta:
                    theta[k] = theta[k] - lr * g[k]
            Xf, yf = stream.eval_on_stage(stage_idx, n=200)
            metrics.append(_eval_metric(theta, Xf, yf, mode))

        records.append({
            "benchmark": benchmark_name, "method": "centralized", "site_id": i, "device_class": dc,
            "round": 20, "fleet_perf_score": float(np.mean(metrics)), "fleet_backward_transfer": None,
            "n_tasks_seen": len(metrics), "commit_count": None, "is_summary_record": True,
        })
    return records

## 10. Three benchmark configurations

Each reuses the identical mechanism above with different `mode` / drift-injection settings, standing in for the manuscript's three real benchmarks:

- **Fed-Twin-CMAPSS**: regression, a subset of sites get a mid-run "recalibration" event (stands in for the FD002/FD004 operating-condition switch).
- **Fed-Twin-FEMTO**: regression, every site drifts (progressive bearing wear, no recalibration event).
- **Fed-Twin-MIMII**: binary classification, a third of sites get a recalibration event (stands in for an SNR/gain change).

In [10]:
BENCHMARK_CONFIGS = {
    "fed-twin-cmapss": dict(mode="regression", stage_len=6, recal_sites=[6, 7, 8, 9, 10]),
    "fed-twin-femto": dict(mode="regression", stage_len=5, recal_sites=None),
    "fed-twin-mimii": dict(mode="classification", stage_len=6, recal_sites=list(range(0, 24, 3))),
}


def run_all_benchmarks(n_rounds=20, local_steps=40, rng_seed=1, **kwargs):
    all_records = []
    for bench_name, cfg in BENCHMARK_CONFIGS.items():
        recs = run_benchmark(bench_name, n_rounds=n_rounds, local_steps=local_steps,
                              rng_seed=rng_seed, **{**cfg, **kwargs})
        all_records.extend(recs)
        all_records.extend(centralized_upper_bound(
            bench_name, cfg["mode"], cfg["stage_len"], rng_seed, recal_sites=cfg["recal_sites"]))
    return all_records


print("Running all three benchmarks x six methods (~2 minutes)...")
records = run_all_benchmarks(n_rounds=20, local_steps=40, rng_seed=RNG_SEED)
print(f"{len(records)} total records")

Running all three benchmarks x six methods (~2 minutes)...


8191 total records


## 11. Results

Mirrors the manuscript's Table 3 schema (Fleet Performance Score and Fleet Backward Transfer per benchmark, per method).

In [11]:
ALL_METHODS = METHODS + ["centralized"]

print(f"{'benchmark':16s} {'method':22s} {'FPS':>8s} {'FBWT':>10s} {'commits':>8s}")
print("-" * 68)
for bench in BENCHMARK_CONFIGS:
    for m in ALL_METHODS:
        summ = [r for r in records if r.get("benchmark") == bench and r.get("method") == m
                and r.get("is_summary_record")]
        fps = np.mean([r["fleet_perf_score"] for r in summ])
        fbwt_vals = [r["fleet_backward_transfer"] for r in summ if r.get("fleet_backward_transfer") is not None]
        fbwt_str = f"{np.mean(fbwt_vals):+.4f}" if fbwt_vals else "n/a"
        commits_vals = [r["commit_count"] for r in summ if r.get("commit_count") is not None]
        commits_str = f"{np.mean(commits_vals):.2f}" if commits_vals else "n/a"
        print(f"{bench:16s} {m:22s} {fps:8.3f} {fbwt_str:>10s} {commits_str:>8s}")
    print()

benchmark        method                      FPS       FBWT  commits
--------------------------------------------------------------------
fed-twin-cmapss  fedavg                   -1.181    -0.0894     0.00
fed-twin-cmapss  fedavg-ewc               -1.109    +0.0159     2.29
fed-twin-cmapss  fedcat-external          -1.292    -0.0244     0.96
fed-twin-cmapss  fedtwin-cl-no-ctfr       -1.504    +0.0000     0.79
fed-twin-cmapss  fedtwin-cl               -1.547    +0.0000     0.79
fed-twin-cmapss  centralized              -0.617        n/a      n/a

fed-twin-femto   fedavg                   -1.235    -0.1588     0.00
fed-twin-femto   fedavg-ewc               -1.180    -0.1246     3.25
fed-twin-femto   fedcat-external          -1.233    -0.0237     0.96
fed-twin-femto   fedtwin-cl-no-ctfr       -1.567    +0.0000     0.79
fed-twin-femto   fedtwin-cl               -1.306    +0.0000     0.79
fed-twin-femto   centralized              -0.615        n/a      n/a

fed-twin-mimii   fedavg         

In [12]:
n_flags = sum(1 for r in records if r.get("ctfr_event"))
n_confirmed = sum(1 for r in records if r.get("ctfr_event") and r.get("ctfr_match") is not None)
print(f"CTFR flags raised across all benchmarks: {n_flags} ({n_confirmed} with an identified match)")

# A quick look at a handful of matches: does the flagged site's fault type
# line up with the site it matched against?
sample = [r for r in records if r.get("ctfr_event")][:5]
for r in sample:
    print(r)

CTFR flags raised across all benchmarks: 559 (559 with an identified match)
{'benchmark': 'fed-twin-cmapss', 'method': 'fedtwin-cl', 'site_id': 3, 'device_class': 'remote-asset', 'round': 3, 'ctfr_event': True, 'ctfr_sim': 0.9571292214572061, 'ctfr_match': (1, 1, 0)}
{'benchmark': 'fed-twin-cmapss', 'method': 'fedtwin-cl', 'site_id': 4, 'device_class': 'remote-asset', 'round': 3, 'ctfr_event': True, 'ctfr_sim': 0.9997467894824988, 'ctfr_match': (1, 1, 0)}
{'benchmark': 'fed-twin-cmapss', 'method': 'fedtwin-cl', 'site_id': 5, 'device_class': 'remote-asset', 'round': 3, 'ctfr_event': True, 'ctfr_sim': 0.9999892659732407, 'ctfr_match': (1, 1, 0)}
{'benchmark': 'fed-twin-cmapss', 'method': 'fedtwin-cl', 'site_id': 7, 'device_class': 'sensor-edge', 'round': 3, 'ctfr_event': True, 'ctfr_sim': 0.9999578669888984, 'ctfr_match': (1, 1, 0)}
{'benchmark': 'fed-twin-cmapss', 'method': 'fedtwin-cl', 'site_id': 9, 'device_class': 'sensor-edge', 'round': 3, 'ctfr_event': True, 'ctfr_sim': 0.998282656

## 12. A few simple plots

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bench in zip(axes, BENCHMARK_CONFIGS):
    for m in METHODS:
        rows = [r for r in records if r.get("benchmark") == bench and r.get("method") == m
                and "raw_metric" in r]
        by_round = {}
        for r in rows:
            by_round.setdefault(r["round"], []).append(r["raw_metric"])
        xs = sorted(by_round)
        ys = [np.mean(by_round[x]) for x in xs]
        ax.plot(xs, ys, label=m, linewidth=1.8)
    ax.set_title(bench)
    ax.set_xlabel("round")
ax.legend(fontsize=7)
axes[0].set_ylabel("current-task metric (higher is better)")
fig.tight_layout()
fig.savefig("spike_fig_convergence.png", dpi=150)
plt.show()
print("Saved spike_fig_convergence.png")

Saved spike_fig_convergence.png


/var/folders/zh/dq8v96hx6rg3l9tyx3bf7_0w0000gn/T/ipykernel_22431/4199823144.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
fig, ax = plt.subplots(figsize=(8, 4.5))
width = 0.14
x = np.arange(len(BENCHMARK_CONFIGS))
for i, m in enumerate(METHODS):
    vals = []
    for bench in BENCHMARK_CONFIGS:
        summ = [r for r in records if r.get("benchmark") == bench and r.get("method") == m
                and r.get("is_summary_record")]
        vals.append(np.mean([r["fleet_backward_transfer"] for r in summ]))
    ax.bar(x + (i - 2) * width, vals, width=width, label=m)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(BENCHMARK_CONFIGS.keys())
ax.set_ylabel("Fleet Backward Transfer (0 = no forgetting)")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig("spike_fig_fbwt.png", dpi=150)
plt.show()
print("Saved spike_fig_fbwt.png -- masking methods (fedcat-external, fedtwin-cl variants) "
      "should sit visibly closer to zero than fedavg/fedavg-ewc on every benchmark.")

Saved spike_fig_fbwt.png -- masking methods (fedcat-external, fedtwin-cl variants) should sit visibly closer to zero than fedavg/fedavg-ewc on every benchmark.


/var/folders/zh/dq8v96hx6rg3l9tyx3bf7_0w0000gn/T/ipykernel_22431/4180575814.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Export

Written as `pilot_results_spike.json`, deliberately **not** named `pilot_results.json` (the name `generate_result_figures.py` expects for the real, dataset-backed pilot) — this is a different thing, and should not be silently swapped in for it. If you want to feed this into `generate_result_figures.py` for a quick look, copy it to `pilot_results.json` yourself, deliberately, and remember it is still synthetic-data output, not the real Section 7 result.

In [15]:
OUT_PATH = "pilot_results_spike.json"
with open(OUT_PATH, "w") as f:
    json.dump(records, f)
print(f"Wrote {len(records)} records to {OUT_PATH}")
print("Reminder: this is SPIKE output (real code, synthetic data), not the real dataset pilot.")

Wrote 8191 records to pilot_results_spike.json
Reminder: this is SPIKE output (real code, synthetic data), not the real dataset pilot.


## 14. What's next: turning this into the real pilot

This notebook validates the *mechanism*. Turning it into the manuscript's actual Section 7 needs, roughly in order:

1. **Real data loaders**, replacing `SiteStream`/`make_fault_library`: NASA C-MAPSS (public, direct download), the IEEE PHM 2012 / FEMTO-ST PRONOSTIA bearing dataset, and MIMII (see `EXPERIMENT_PROCEDURE.md` for sources and preprocessing). The health-indicator computation per benchmark (RUL-residual proxy, RMS-of-vibration-envelope, rolling anomaly score) needs to be built from the real signal, not simulated.
2. **Real architectures**, replacing the single-hidden-layer MLP: TCN-Nano for the two regression benchmarks, DS-CNN-S for the acoustic benchmark, per Section 6.2 of the manuscript — likely in PyTorch, given real conv architectures and larger data volumes.
3. **A genuine cross-site federation story for still-free (uncommitted) filters.** This notebook deliberately does NOT pool free-space training across sites (Section 7's aggregation-design note explains why, given synthetic sites are frequently on unrelated concurrent tasks); with real sensor data from physically similar equipment, sites' concurrent tasks are far more likely to be genuinely related, and it is worth revisiting whether real cross-site averaging of free-space gradients helps rather than interferes.
4. **FedProx**, omitted here for scope — a straightforward addition (FedAvg plus a proximal term in the local loss).
5. **CTFR precision tuning.** This notebook's CTFR raises a fairly large number of flags at a high similarity threshold; a real signature (built from real health-indicator features, not the current six summary statistics) will need its own precision/recall validation, mirroring Table 6's flag-precision metric, before its lead-time numbers mean anything.
6. Once real numbers exist, export in exactly this notebook's record schema and feed straight into `generate_result_figures.py` (`FedTwin-CL/generate_result_figures.py`) to produce the real Figures 5-10 and replace every `[HYPOTHETICAL]` table cell in the manuscript.